1. BPE (Byte-Pair Encoding): The Frequency King
    - How it works: It starts by splitting all text into single characters (or bytes). It then looks at the training corpus and finds the most frequently adjacent pair of characters (e.g., "t" and "h"). It merges them into a single token ("th"). It repeats this process iteratively until it hits a predetermined vocabulary size (e.g., 50,257 for GPT-2).
    - Used by: OpenAI models (GPT-2, GPT-3, GPT-4 via tiktoken), Meta's RoBERTa.

2. WordPiece: The Probabilistic Optimizer
    - How it works: Very similar to BPE, but instead of merging the most frequent pair, it merges the pair that most maximizes the likelihood of the training data. It evaluates pairs based on the formula: $Score = \frac{freq(AB)}{freq(A) \times freq(B)}$. This means it favors combining rare tokens that frequently appear together over common tokens that just happen to be next to each other.
    - Used by: Google's BERT, Electra.

3. SentencePiece: The Language-Agnostic Wrapper
    - How it works: SentencePiece isn't actually a distinct splitting algorithm (it usually uses BPE or a Unigram model under the hood). Instead, it's a preprocessing philosophy. Traditional tokenizers require text to be split by spaces first (pre-tokenization). SentencePiece treats spaces as just another character (represented visually as _ or   ). This is crucial for languages like Chinese or Japanese that don't use spaces.
    - Used by: Meta's LLaMA, Google's T5, ALBERT.

---

#### Basic BPE Tokenizer

In [3]:
import re
from collections import Counter, defaultdict

fd = open('mock_text.txt', 'r')
text = fd.read()

# Remove punctuation and make lowercase
text = re.sub(r'[^\w\s]', '', text).lower()

# Split text into words
words = text.split()

# Count word frequencies

word_counts = Counter(words)

# Print the 10 most common words
print(word_counts.most_common(10))

# Create a dictionary to hold word frequencies
word_freq = defaultdict(int)
for word in words:
    word_freq[word] += 1

vocab = {}
for word, freq in word_freq.items():
    spaced_word = ' '.join(list(word)) + ' </w>'
    vocab[spaced_word] = freq

def get_stats(vocab):
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pair = (symbols[i], symbols[i + 1])
            pairs[pair] += freq
    return pairs

def merge_vocab(pair, vocab):
    new_vocab = {}
    bigram = ' '.join(pair)
    replacement = ''.join(pair)
    for word in vocab:
        new_word = word.replace(bigram, replacement)
        new_vocab[new_word] = vocab[word]
    return new_vocab

num_merges = 100
print("Initial Vocabulary:", vocab)

for i in range(num_merges):
    pairs = get_stats(vocab)
    if not pairs:
        break
    best_pairs = max(pairs, key=pairs.get)
    vocab = merge_vocab(best_pairs, vocab)
    print(f"Merge {i + 1}: {best_pairs} -> {vocab}")
    print(f'current vocab: {vocab}\n')

[('the', 12), ('and', 9), ('a', 6), ('token', 4), ('quick', 3), ('words', 3), ('subwords', 3), ('are', 3), ('cat', 3), ('like', 3)]
Initial Vocabulary: {'t h e </w>': 12, 'q u i c k </w>': 3, 'b r o w n </w>': 2, 'f o x </w>': 2, 'j u m p s </w>': 1, 'o v e r </w>': 1, 'l a z y </w>': 1, 'd o g </w>': 2, 'w a s </w>': 2, 'v e r y </w>': 1, 'a </w>': 6, 't o k e n i z e r </w>': 2, 's p l i t s </w>': 1, 't e x t </w>': 2, 'i n t o </w>': 2, 's m a l l e r </w>': 2, 'p i e c e s </w>': 1, 't o k e n i z a t i o n </w>': 2, 'h e l p s </w>': 1, 'm o d e l s </w>': 1, 'h a n d l e </w>': 1, 'r a r e </w>': 2, 'w o r d s </w>': 3, 'c a n </w>': 2, 'b e </w>': 2, 's p l i t </w>': 2, 's u b w o r d s </w>': 3, 'a r e </w>': 3, 'u s e f u l </w>': 1, 'f o r </w>': 2, 'u n s e e n </w>': 1, 't e r m s </w>': 1, 's e n t e n c e p i e c e </w>': 1, 't r a i n </w>': 1, 'f r o m </w>': 1, 'r a w </w>': 1, 'b p e </w>': 1, 'm e r g e s </w>': 1, 'f r e q u e n t </w>': 1, 'c h a r a c t e r </w>

---

#### Basic WordPiece Implementation

In [4]:
import re
from collections import Counter, defaultdict

fd = open('mock_text.txt', 'r')
text = fd.read()

# Remove punctuation and make lowercase
text = re.sub(r'[^\w\s]', '', text).lower()

# Split text into words
words = text.split()

# Count word frequencies

word_counts = Counter(words)

# Print the 10 most common words
print(word_counts.most_common(10))

# Create a dictionary to hold word frequencies
word_freq = defaultdict(int)
for word in words:
    word_freq[word] += 1

vocab = {}
for word, freq in word_freq.items():
    if not word:
        continue
    chars = [word[0]] + ['##' + c for c in word[1:]]
    vocab[' '.join(chars)] = freq

def get_stats(vocab):
    pairs = defaultdict(int)
    freqs = defaultdict(int)

    for word, freq in vocab.items():
        symbols = word.split()
        for i, sym in enumerate(symbols):
            freqs[sym] += freq
            if i < len(symbols) - 1:
                pairs[(sym, symbols[i + 1])] += freq
            
    return pairs, freqs

def merge_vocab(pair, v_in):
    v_out = {}
    token_a, token_b = pair
    merged_token = token_a + token_b.replace('##', '', 1)

    bigram = re.escape(' '.join(pair))
    p = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')

    for word in v_in:
        w_out = p.sub(merged_token, word)
        v_out[w_out] = v_in[word]
    return v_out

num_merges = 100
print("Initial Vocabulary:")
for word, freq in vocab.items():
    print(f"{word}: {freq}")

for i in range(num_merges):
    pairs, freqs = get_stats(vocab)
    if not pairs:
        break
    
    scores = {}
    for pair, pair_freq in pairs.items():
        scores[pair] = pair_freq / (freqs[pair[0]] * freqs[pair[1]])
    
    best_pair = max(scores, key=scores.get)
    vocab = merge_vocab(best_pair, vocab)
    print(f"Merge {i + 1}: {best_pair} -> {vocab}")
    print(f'current vocab: {list(vocab.keys())}\n')




[('the', 12), ('and', 9), ('a', 6), ('token', 4), ('quick', 3), ('words', 3), ('subwords', 3), ('are', 3), ('cat', 3), ('like', 3)]
Initial Vocabulary:
t ##h ##e: 12
q ##u ##i ##c ##k: 3
b ##r ##o ##w ##n: 2
f ##o ##x: 2
j ##u ##m ##p ##s: 1
o ##v ##e ##r: 1
l ##a ##z ##y: 1
d ##o ##g: 2
w ##a ##s: 2
v ##e ##r ##y: 1
a: 6
t ##o ##k ##e ##n ##i ##z ##e ##r: 2
s ##p ##l ##i ##t ##s: 1
t ##e ##x ##t: 2
i ##n ##t ##o: 2
s ##m ##a ##l ##l ##e ##r: 2
p ##i ##e ##c ##e ##s: 1
t ##o ##k ##e ##n ##i ##z ##a ##t ##i ##o ##n: 2
h ##e ##l ##p ##s: 1
m ##o ##d ##e ##l ##s: 1
h ##a ##n ##d ##l ##e: 1
r ##a ##r ##e: 2
w ##o ##r ##d ##s: 3
c ##a ##n: 2
b ##e: 2
s ##p ##l ##i ##t: 2
s ##u ##b ##w ##o ##r ##d ##s: 3
a ##r ##e: 3
u ##s ##e ##f ##u ##l: 1
f ##o ##r: 2
u ##n ##s ##e ##e ##n: 1
t ##e ##r ##m ##s: 1
s ##e ##n ##t ##e ##n ##c ##e ##p ##i ##e ##c ##e: 1
t ##r ##a ##i ##n: 1
f ##r ##o ##m: 1
r ##a ##w: 1
b ##p ##e: 1
m ##e ##r ##g ##e ##s: 1
f ##r ##e ##q ##u ##e ##n ##t: 1
c ##h ##a ##r ##a ##

---

#### Basic SentencePiece Implementation

In [11]:
from collections import defaultdict

# The iconic SentencePiece meta-symbol
SPIECE_UNDERLINE = ' ' 

# 1. Start with raw sentences, NOT word dictionaries.
# Notice the awkward double space in the second sentence. 
# Standard tokenizers would lose that double space. SentencePiece won't.
raw_sentences = [
    "Tokenization is incredibly pseudoscientific and antiestablishmentarianism-esque.",
    "hello  world",
]

# 2. The Language-Agnostic Wrapper: Treat spaces as characters
# We replace normal spaces with the meta-symbol, then split into characters.
corpus = []
for sentence in raw_sentences:
    # Replace space with the special symbol
    processed = sentence.replace(" ", SPIECE_UNDERLINE)
    # Split into a list of individual characters
    corpus.append(list(processed))

def get_stats(corpus):
    """Count frequencies of adjacent pairs across all sentences."""
    pairs = defaultdict(int)
    for sentence in corpus:
        for i in range(len(sentence) - 1):
            # We count pairs natively, whether they involve letters or spaces
            pairs[(sentence[i], sentence[i+1])] += 1
    return pairs

def merge_vocab(pair, corpus):
    """Merge the target pair across all sentences in the corpus."""
    new_corpus = []
    for sentence in corpus:
        i = 0
        new_sentence = []
        while i < len(sentence):
            # If we find the pair, merge them into a single string
            if i < len(sentence) - 1 and sentence[i] == pair[0] and sentence[i+1] == pair[1]:
                new_sentence.append(pair[0] + pair[1])
                i += 2
            else:
                new_sentence.append(sentence[i])
                i += 1
        new_corpus.append(new_sentence)
    return new_corpus

# 3. Train the SentencePiece BPE model
num_merges = 100

print("--- Training SentencePiece ---")
for i in range(num_merges):
    pairs = get_stats(corpus)
    if not pairs:
        break
        
    best_pair = max(pairs, key=pairs.get)
    corpus = merge_vocab(best_pair, corpus)
    
    print(f"Iteration {i+1}: Merging {best_pair}")

print("\n--- Final Tokenized Corpus ---")
for i, tokens in enumerate(corpus):
    print(f"Sentence {i+1}: {tokens}")

# 4. The Magic of Lossless Decoding
def decode(tokens):
    """Reconstruct the exact original text without guessing about spaces."""
    # Step A: Concatenate all tokens together
    joined_text = "".join(tokens)
    # Step B: Swap the meta-symbol back to a standard space
    return joined_text.replace(SPIECE_UNDERLINE, " ")

print("\n--- Lossless Decoding Test ---")
test_tokens = corpus[1] # The sentence with the double space
reconstructed = decode(test_tokens)
print(f"Tokens: {test_tokens}")
print(f"Decoded string: '{reconstructed}'")
print(f"Matches original exactly? {reconstructed == raw_sentences[1]}")

--- Training SentencePiece ---
Iteration 1: Merging ('e', 'n')
Iteration 2: Merging ('t', 'i')
Iteration 3: Merging ('i', 's')
Iteration 4: Merging ('a', 'n')
Iteration 5: Merging ('b', 'l')
Iteration 6: Merging (' ', 'an')
Iteration 7: Merging ('e', 's')
Iteration 8: Merging ('t', 'a')
Iteration 9: Merging ('T', 'o')
Iteration 10: Merging ('To', 'k')
Iteration 11: Merging ('Tok', 'en')
Iteration 12: Merging ('Token', 'i')
Iteration 13: Merging ('Tokeni', 'z')
Iteration 14: Merging ('Tokeniz', 'a')
Iteration 15: Merging ('Tokeniza', 'ti')
Iteration 16: Merging ('Tokenizati', 'o')
Iteration 17: Merging ('Tokenizatio', 'n')
Iteration 18: Merging ('Tokenization', ' ')
Iteration 19: Merging ('Tokenization ', 'is')
Iteration 20: Merging ('Tokenization is', ' ')
Iteration 21: Merging ('Tokenization is ', 'i')
Iteration 22: Merging ('Tokenization is i', 'n')
Iteration 23: Merging ('Tokenization is in', 'c')
Iteration 24: Merging ('Tokenization is inc', 'r')
Iteration 25: Merging ('Tokenizatio

---
#### Experiment with Tokenization Strategies

In [6]:
import tiktoken
from transformers import BertTokenizer, LlamaTokenizerFast

# Load Tokenizers
# 1. BPE via OpenAI
gpt_tokenizer = tiktoken.get_encoding("cl100k_base") # Used by GPT-4

# 2. WordPiece via BERT
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# 3. SentencePiece (BPE under the hood) via LLaMA 
# (Using a standard HF fast tokenizer for LLaMA as a proxy)
try:
    llama_tokenizer = LlamaTokenizerFast.from_pretrained("hf-internal-testing/llama-tokenizer")
except:
    print("Using BERT as fallback if Llama requires HF login.")

# The Experiment
text = "Tokenization is incredibly pseudoscientific and antiestablishmentarianism-esque."

print("--- Original Text ---")
print(text)

print("\n--- 1. GPT-4 (BPE) ---")
gpt_tokens = gpt_tokenizer.encode(text)
print([gpt_tokenizer.decode([t]) for t in gpt_tokens])
print(f"Token count: {len(gpt_tokens)}")

print("\n--- 2. BERT (WordPiece) ---")
bert_tokens = bert_tokenizer.tokenize(text)
print(bert_tokens)
print(f"Token count: {len(bert_tokens)}")

print("\n--- 3. LLaMA (SentencePiece) ---")
try:
    llama_tokens = llama_tokenizer.tokenize(text)
    print(llama_tokens)
    print(f"Token count: {len(llama_tokens)}")
except:
    pass

tokenizer_config.json: 0.00B [00:00, ?B/s]

c:\MIX\ASU\SEM_4\LLM_Basics\phase2\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\amogh\.cache\huggingface\hub\models--hf-internal-testing--llama-tokenizer. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

--- Original Text ---
Tokenization is incredibly pseudoscientific and antiestablishmentarianism-esque.

--- 1. GPT-4 (BPE) ---
['Token', 'ization', ' is', ' incredibly', ' pseud', 'os', 'cient', 'ific', ' and', ' anti', 'establish', 'ment', 'arian', 'ism', '-esque', '.']
Token count: 16

--- 2. BERT (WordPiece) ---
['token', '##ization', 'is', 'incredibly', 'pseudo', '##sc', '##ient', '##ific', 'and', 'anti', '##est', '##ab', '##lish', '##ment', '##arian', '##ism', '-', 'esq', '##ue', '.']
Token count: 20

--- 3. LLaMA (SentencePiece) ---
['▁Token', 'ization', '▁is', '▁incred', 'ibly', '▁pseud', 'os', 'cient', 'ific', '▁and', '▁anti', 'est', 'ab', 'lish', 'ment', 'arian', 'ism', '-', 'es', 'que', '.']
Token count: 21


In [1]:
import tiktoken
from transformers import BertTokenizer, LlamaTokenizerFast

# Load Tokenizers
# 1. BPE via OpenAI
gpt_tokenizer = tiktoken.get_encoding("cl100k_base") # Used by GPT-4

# 2. WordPiece via BERT
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# 3. SentencePiece (BPE under the hood) via LLaMA 
# (Using a standard HF fast tokenizer for LLaMA as a proxy)
try:
    llama_tokenizer = LlamaTokenizerFast.from_pretrained("hf-internal-testing/llama-tokenizer")
except:
    print("Using BERT as fallback if Llama requires HF login.")

# The Experiment
file = open('mock_text.txt', 'r')
text = file.read()
file.close()

print("--- Original Text ---")
print(text)

print("\n--- 1. GPT-4 (BPE) ---")
gpt_tokens = gpt_tokenizer.encode(text)
print([gpt_tokenizer.decode([t]) for t in gpt_tokens])
print(f"Token count: {len(gpt_tokens)}")

print("\n--- 2. BERT (WordPiece) ---")
bert_tokens = bert_tokenizer.tokenize(text)
print(bert_tokens)
print(f"Token count: {len(bert_tokens)}")

print("\n--- 3. LLaMA (SentencePiece) ---")
try:
    llama_tokens = llama_tokenizer.tokenize(text)
    print(llama_tokens)
    print(f"Token count: {len(llama_tokens)}")
except:
    pass

--- Original Text ---
The quick brown fox jumps over the lazy dog.
The quick brown fox was very quick.
A tokenizer splits text into smaller pieces.
Tokenization helps models handle rare words.
Rare words can be split into subwords.
Subwords are useful for unseen terms.
SentencePiece can train from raw text.
BPE merges frequent character pairs.
WordPiece learns subwords with likelihood-based choices.
I love building NLP tools in Python.

The cat sat on the mat.
The cat did not like the mat.
A dog chased the cat around the yard.
The yard was quiet and sunny.
Sunny days make people smile.
People like simple examples for learning.

He said, "Hello!" and waved.
Don't split contractions too aggressively.
Numbers like 123, 456, and 789 should stay meaningful.
Email addresses such as test@example.com are tricky.
Newlines, tabs, and    extra spaces should be normalized.

token token token token.
tokenization tokenized tokenizing tokenizer.
play played playing player.
run ran running runner.
jum



1. How It Works in the Code
    - If you look back at the BPE script we wrote earlier, you will see a variable called `num_merges = 10`. That is a micro-version of a hard-coded limit. 

    - In a real training scenario, the tokenizer starts with individual characters (a vocabulary of maybe 256 bytes). It merges the most frequent pair, making the vocabulary 257. It merges the next pair, making it 258. It repeats this loop until it precisely hits the hard-coded limit (e.g., 50,000). At that exact moment, the tokenizer freezes its vocabulary forever. 

2. Why not make the limit infinite? (The Matrix Problem)
    - You might wonder: *Why not just let the model learn 10 million tokens so it knows every word, name, and number perfectly?* Because of **The Embedding Matrix**. 
    - Inside the Large Language Model, every single token in the vocabulary gets its own dedicated row in a massive mathematical lookup table (the embedding matrix). This row contains thousands of numbers that represent the "meaning" of that token.
        - If you have a vocabulary of 50,000 tokens, your model has to store 50,000 rows of weights in its RAM.
        - If you increase the vocabulary to 10 million tokens, the model's memory requirements would explode. The model would become too massive to fit on a GPU, and inference would slow to a crawl. 

3. The "Goldilocks" Trade-off
    - Setting this hard limit is a balancing act. Engineers must choose a number that is "just right" to balance compression and flexibility.
    - Limit is too small (e.g., 200 tokens): The model only knows basic letters and symbols.
        - Pros: The model uses almost zero memory. 
        - Cons: It takes 15 tokens just to spell "pseudoscientific." The model's context window will fill up immediately, and it will be agonizingly slow at generating text because it has to predict text one letter at a time.
    - Limit is too large (e.g., 1,000,000 tokens): The model memorizes whole words and even entire common phrases.
        - Pros: Highly compressed. "Pseudoscientific" is just 1 token.
        - Cons: The model becomes massively bloated and requires absurd amounts of VRAM. Worse, it becomes rigid. If a user makes a typo ("psuedoscientific"), the model won't have a token for it and will panic (yielding an "Out of Vocabulary" error or outputting garbage).
    - The Sweet Spot (30,000 to 100,000 tokens):
        - This forces the tokenizer to learn common whole words (like "apple", "the", "code"), but forces it to keep powerful prefixes and suffixes (like "pseudo", "anti", "ing", "ly") to build rare or made-up words dynamically.